# KIZUNA – train the vision models (Google Colab, free GPU)

**Runtime → Change runtime type → T4 GPU**, then run the cells in order.

1. **Pest / damage detector** (YOLO object detection): *Tuta absoluta* mines and larvae, whiteflies, aphids… → `pest.pt`, used by the vision service as `PEST_MODEL`.
2. **Crop + growth-stage classifier** (YOLO classification) from your own field photos → `crop.pt`, used as `CROP_MODEL`. It replaces the zero-shot prototype for crop and stage.

The irrigation volume is never learned here: these models only describe what the camera sees.

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics; ultralytics.checks()

## 1. Pest / damage detector

### Option A – start from a public dataset (Roboflow)
Create a free Roboflow account, copy your API key (Settings → API keys), then run the cell.
The example is the public *TutaAbsolute-Det-V2* dataset; you can merge it with your own photos later in Roboflow.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="PASTE_YOUR_ROBOFLOW_KEY")
ds = rf.workspace("leaf-miner-dataset").project("tutaabsolute-det-v2").version(4).download("yolov8")
DATA_YAML = ds.location + "/data.yaml"
print(open(DATA_YAML).read())

### Option B – your own labelled photos
Label them in Roboflow or CVAT (free), export **YOLOv8** format as a zip, upload it to Google Drive, then:

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !unzip -q /content/drive/MyDrive/kizuna_pests.zip -d /content/pests
# DATA_YAML = '/content/pests/data.yaml'

### Train
`yolov8n` is small enough for a laptop or a Raspberry Pi 5; use `yolov8s` if accuracy is too low and you have a PC.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(data=DATA_YAML, epochs=80, imgsz=640, patience=20, batch=16, project="kizuna", name="pest")

### Check the accuracy on images it has never seen
Look at **mAP50** per class and the confusion matrix. Report these numbers in the pitch – they are your measured accuracy.

In [ ]:
best = YOLO("kizuna/pest/weights/best.pt")
metrics = best.val(data=DATA_YAML, split="test" if "test:" in open(DATA_YAML).read() else "val")
print("mAP50:", round(metrics.box.map50, 3), " mAP50-95:", round(metrics.box.map, 3))
for i, c in enumerate(metrics.box.ap_class_index):
    print(f"{best.names[int(c)]:25s} AP50 = {metrics.box.ap50[i]:.3f}")

In [ ]:
from IPython.display import Image, display
display(Image("kizuna/pest/confusion_matrix.png", width=600))
res = best.predict(source=ds.location + "/test/images" if "ds" in globals() else "/content/pests/test/images",
                   conf=0.35, save=True, max_det=50)
print("Predictions saved in", res[0].save_dir)

In [ ]:
from google.colab import files
!cp kizuna/pest/weights/best.pt pest.pt
files.download("pest.pt")   # put it in kizuna-vision/models/ and set PEST_MODEL=models/pest.pt

## 2. Crop + growth-stage classifier (your own photos)

Folder layout (one folder per class, name = `crop-stage`, stage names as in FAO-56):
```
crop_stage/
  train/  tomato-initial/  tomato-development/  tomato-mid_season/  tomato-late_season/
          pepper-initial/  pepper-development/  pepper-mid_season/  pepper-late_season/
          potato-…/        eggplant-…/       (every crop your farmers grow, same 4 stages)
          other-none/      (weeds, bare soil, irrigation pipes…)
  val/    same folders, different plants
```
Zip `crop_stage`, put it in Google Drive.

**Look-alike crops** (tomato, pepper, potato, eggplant are all in the same plant family): put
the same number of photos in each, include close-ups of the leaves, and check the confusion
matrix – if potato is still predicted as pepper, add more potato *and* pepper photos taken
the same way.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !unzip -q /content/drive/MyDrive/crop_stage.zip -d /content
CLS_DATA = "/content/crop_stage"
cls = YOLO("yolov8n-cls.pt")
cls.train(data=CLS_DATA, epochs=40, imgsz=224, patience=10, project="kizuna", name="crop")

In [ ]:
best_cls = YOLO("kizuna/crop/weights/best.pt")
m = best_cls.val(data=CLS_DATA)
print("top-1 accuracy:", round(m.top1, 3))
display(Image("kizuna/crop/confusion_matrix.png", width=600))
!cp kizuna/crop/weights/best.pt crop.pt
files.download("crop.pt")   # kizuna-vision/models/crop.pt, set CROP_MODEL=models/crop.pt